In [1]:
!pip -q install lightgbm==4.6.0
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
import numpy, pandas, lightgbm
print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| lightgbm', lightgbm.__version__)
try:
    import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
except ImportError:
    print('torch not installed: the MLP runs need it')

Mounted at /content/drive
pandas 2.2.3 | numpy 2.4.6 | lightgbm 4.6.0
torch 2.14.1+cu130 | cuda True


In [2]:
# Experiment A: random-row (RR) versus hash-group-disjoint (GD) on the same rows.
# Cohorts: frozen (attack-enriched subset of step 5) and full (every row of the dataset).
# Matrix: split_seed x model_seed (Cartesian), LightGBM and MLP on frozen, LightGBM on full.
SMOKE = False                      # True: one dataset, one split seed, one model seed, short budgets

DATA_DIR    = '/content/drive/MyDrive'
RUN_ROOT    = '/content/drive/MyDrive/THAT_IDS_RERUN'
SUBSET_ROOT = f'{RUN_ROOT}/conference_step5_literature_subset_v1/subset_definition'
OUT_DIR     = f'{RUN_ROOT}/journal_step22_rr_vs_gd_v1'

DATASETS_TO_RUN = ['UNSW_V2', 'CIC_V2']
SPLIT_SEEDS = [42, 123, 456]
MODEL_SEEDS = [42, 123, 456]
PROTOCOLS = ['random_row', 'hash_group_disjoint']
PLAN = {                           # cohort -> models (the plan's run matrix)
    'frozen': ['lightgbm', 'mlp'],
    'full':   ['lightgbm'],
}
FULL_GD_FROM_PAPER_SALTS = True    # rebuild the six stored full-data GD splits from their salts
N_CANDIDATES_NEW_SPLITS = 24       # Algorithm 1 candidates for the new frozen-cohort splits
CLIP = None                        # None = no clipping after MinMax (plan recommends clip=False); (-10, 10) = paper's neural pipeline
FPR_TARGETS = (0.001, 0.0001)      # 0.1 % and 0.01 %; the 0.01 % point is flagged when validation is too small

LGBM_PARAMS = dict(objective='binary', n_estimators=2000, learning_rate=0.03, num_leaves=31,
                   max_depth=-1, min_child_samples=50, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0, max_bin=255,
                   deterministic=True, force_col_wise=True, n_jobs=-1, verbose=-1)
LGBM_EARLY_STOP = 100
MLP_CFG = dict(enc_hidden=192, latent=96, cls_hidden=(192, 96), enc_dropout=0.10, cls_dropout=0.15,
               lr=5e-5, weight_decay=1e-5, grad_clip=1.0, max_epochs=80, patience=12, batch=5000,
               focal_alpha=0.75, focal_gamma=2.0, pos_weight_cap=12.0)

if SMOKE:
    DATASETS_TO_RUN = ['UNSW_V2']; SPLIT_SEEDS = [42]; MODEL_SEEDS = [42]
    LGBM_PARAMS['n_estimators'] = 200; MLP_CFG['max_epochs'] = 3
    OUT_DIR += '_smoke'
os.makedirs(OUT_DIR, exist_ok=True)
for sub in ('splits', 'runs', 'predictions', 'reports'):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
print('output:', OUT_DIR, '| smoke:', SMOKE)

output: /content/drive/MyDrive/THAT_IDS_RERUN/journal_step22_rr_vs_gd_v1 | smoke: False


In [3]:
# ---- Shared code: data loading, hash groups, splits, thresholds, metrics, bootstrap ----
# Copied verbatim into each experiment notebook so that every notebook is self-contained.
import os, json, time, gc, math, hashlib, warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score

EXCLUDE_COLS = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
                'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'Label', 'Attack']
N_FEATURES = 39
CHUNK = 1_000_000
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10
TARGET_SPLIT_FRACTIONS = np.asarray([0.70, 0.10, 0.20], dtype=np.float64)
N_CANDIDATE_SALTS = 24
MIN_FAMILY_TOTAL_FOR_BALANCE = 30
SHUFFLE_SEED_OFFSET = 17_171
SPLIT_NAMES = ('train', 'validation', 'test')

DATASETS = {
    'UNSW_V2': dict(file='NF-UNSW-NB15-v2.csv', rows=2_390_275, groups=236_999,
                    frozen_rows=245_994, frozen_groups=41_640),
    'CIC_V2':  dict(file='NF-CSE-CIC-IDS2018-v2.csv', rows=18_893_708, groups=8_981_062,
                    frozen_rows=496_384, frozen_groups=233_966),
}
# Salts and (rows, benign, attack) of the six stored full-data group-disjoint splits
# (splits/paper_full_data_splits.json in the published package).
PAPER_FULL_SPLITS = {
    ('UNSW_V2', 42):  dict(salt=11652151511078254891,
        train=(1670959, 1604425, 66534), validation=(248331, 235791, 12540), test=(470985, 455006, 15979)),
    ('UNSW_V2', 123): dict(salt=4518799303034446728,
        train=(1669960, 1600374, 69586), validation=(219285, 210798, 8487), test=(501030, 484050, 16980)),
    ('UNSW_V2', 456): dict(salt=5346391484498864780,
        train=(1667966, 1600280, 67686), validation=(269673, 260319, 9354), test=(452636, 434623, 18013)),
    ('CIC_V2', 42):   dict(salt=8095878257575067585,
        train=(13232623, 11630102, 1602521), validation=(2012758, 1796541, 216217), test=(3648327, 3208924, 439403)),
    ('CIC_V2', 123):  dict(salt=9558457822766297306,
        train=(13254901, 11705849, 1549052), validation=(1893721, 1659711, 234010), test=(3745086, 3270007, 475079)),
    ('CIC_V2', 456):  dict(salt=3253095538486194220,
        train=(13336590, 11700738, 1635852), validation=(1836131, 1621337, 214794), test=(3720987, 3313492, 407495)),
}


def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)


def atomic_json(obj, path):
    tmp = str(path) + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    os.replace(tmp, path)


# ---------------------------------------------------------------- data ----
def load_dataset(name, data_dir):
    """Chunked reader. Returns X (float32, 39 cols), uint64 group keys, binary labels,
    family codes, family names and the replaced-value count per column."""
    cfg = DATASETS[name]
    path = os.path.join(data_dir, cfg['file'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path}: edit DATA_DIR in the config cell')
    header = pd.read_csv(path, nrows=0).columns.tolist()
    feat_cols = [c for c in header if c not in EXCLUDE_COLS]
    assert len(feat_cols) == N_FEATURES, f'{len(feat_cols)} feature columns, expected {N_FEATURES}'
    n = cfg['rows']
    X = np.empty((n, N_FEATURES), dtype=np.float32)
    keys = np.empty(n, dtype=np.uint64)
    y = np.empty(n, dtype=np.int8)
    fam = np.empty(n, dtype=np.int16)
    replaced = np.zeros(N_FEATURES, dtype=np.int64)
    fam_names, pos, t0 = {}, 0, time.time()
    for chunk in pd.read_csv(path, usecols=feat_cols + ['Label', 'Attack'], chunksize=CHUNK):
        f = chunk[feat_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
        a = f.to_numpy(copy=True)
        bad = ~np.isfinite(a)
        replaced += bad.sum(axis=0)
        a[bad] = 0.0
        f = pd.DataFrame(a, columns=feat_cols)
        m = len(f)
        if pos + m > n:
            raise ValueError(f'{name}: more rows than expected ({n})')
        X[pos:pos + m] = a
        keys[pos:pos + m] = pd.util.hash_pandas_object(f, index=False).to_numpy()
        y[pos:pos + m] = chunk['Label'].to_numpy().astype(np.int8)
        att = chunk['Attack'].astype(str)
        for v in att.unique():
            fam_names.setdefault(v, len(fam_names))
        fam[pos:pos + m] = att.map(fam_names).to_numpy().astype(np.int16)
        pos += m
        print(f'\r{name}: {pos:,} rows read ({time.time() - t0:.0f}s)', end='')
    print()
    assert pos == n, f'{name}: read {pos:,} rows, expected {n:,}'
    n_groups = len(np.unique(keys))
    log(f'{name}: groups {n_groups:,} (audit {cfg["groups"]:,}) | repeated-row rate '
        f'{100 * (1 - n_groups / n):.2f}% | attack {100 * y.mean():.2f}% | replaced values {int(replaced.sum()):,}')
    if n_groups != cfg['groups']:
        print('  WARNING: group count differs from the audit; preprocessing does not match the paper.')
    names = [k for k, _ in sorted(fam_names.items(), key=lambda kv: kv[1])]
    return dict(name=name, X=X, keys=keys, y=y, fam=fam, names=np.array(names), feat_cols=feat_cols,
                n_groups=n_groups, replaced_per_column=dict(zip(feat_cols, replaced.tolist())))


def load_frozen_cohort(name, data, subset_root):
    """Row indices of the frozen attack-enriched subset (subset_indices_int32.npy of step 5)."""
    cfg = DATASETS[name]
    path = os.path.join(subset_root, name, 'subset_indices_int32.npy')
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path}: the frozen subset definition is needed for the frozen cohort')
    idx = np.load(path).astype(np.int64)
    assert len(idx) == cfg['frozen_rows'], f'{name}: subset has {len(idx):,} rows, expected {cfg["frozen_rows"]:,}'
    assert len(np.unique(idx)) == len(idx), 'duplicate row indices in the subset'
    sub_keys = data['keys'][idx]
    n_groups = len(np.unique(sub_keys))
    assert n_groups == cfg['frozen_groups'], f'{name}: subset has {n_groups:,} groups, expected {cfg["frozen_groups"]:,}'
    # complete-group check: every group of the subset must be entirely inside it
    full_counts = pd.Series(data['keys']).value_counts()
    sub_counts = pd.Series(sub_keys).value_counts()
    assert (full_counts.loc[sub_counts.index].to_numpy() == sub_counts.to_numpy()).all(), 'subset contains partial groups'
    log(f'{name}: frozen cohort {len(idx):,} rows, {n_groups:,} complete groups, attack {100 * data["y"][idx].mean():.2f}%')
    return idx


# --------------------------------------------------------------- splits ----
def splitmix64(values):
    x = np.asarray(values, dtype=np.uint64).copy()
    with np.errstate(over='ignore'):
        x = x + np.uint64(0x9E3779B97F4A7C15)
        x = (x ^ (x >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        x = (x ^ (x >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        x = x ^ (x >> np.uint64(31))
    return x


def assign_groups(unique_hashes, salt):
    mixed = splitmix64(np.asarray(unique_hashes) ^ np.uint64(salt))
    unit = mixed.astype(np.float64) / float(2 ** 64)
    return np.where(unit < TRAIN_FRAC, 0, np.where(unit < TRAIN_FRAC + VAL_FRAC, 1, 2)).astype(np.int8)


def assignment_objective(group_assignment, group_inverse, group_counts, family_codes, n_families):
    counts = np.asarray(group_counts, dtype=np.int64)
    split_rows = np.bincount(group_assignment.astype(np.int64), weights=counts, minlength=3).astype(np.int64)
    split_code = group_assignment[np.asarray(group_inverse)].astype(np.int64)
    combined = split_code * n_families + np.asarray(family_codes).astype(np.int64)
    split_family = np.bincount(combined, minlength=3 * n_families).reshape(3, n_families)
    row_fractions = split_rows / max(split_rows.sum(), 1)
    global_error = float(np.abs(row_fractions - TARGET_SPLIT_FRACTIONS).mean())
    totals = split_family.sum(axis=0)
    eligible = totals >= MIN_FAMILY_TOTAL_FOR_BALANCE
    family_fraction = np.divide(split_family, totals[None, :], out=np.zeros_like(split_family, dtype=np.float64),
                                where=totals[None, :] > 0)
    family_error = np.abs(family_fraction - TARGET_SPLIT_FRACTIONS[:, None])
    if eligible.any():
        weights = np.sqrt(totals[eligible].astype(np.float64)); weights /= weights.sum()
        family_mae = float((family_error[:, eligible].mean(axis=0) * weights).sum())
        max_family_error = float(family_error[:, eligible].max())
    else:
        family_mae = max_family_error = 0.0
    expected = TARGET_SPLIT_FRACTIONS[:, None] * totals[None, :]
    missing_required = int(((split_family == 0) & (expected >= 5)).sum())
    objective = 5.0 * global_error + family_mae + 0.25 * max_family_error + 10.0 * missing_required
    return dict(objective=float(objective), global_fraction_mae=global_error, family_fraction_weighted_mae=family_mae,
                max_family_fraction_error=max_family_error, missing_required_family_cells=missing_required,
                split_rows=split_rows.tolist())


def hash_group_split(keys, family_codes, seed, n_candidates=N_CANDIDATE_SALTS, salt=None):
    """Algorithm 1. Returns part (0 train / 1 val / 2 test per row, in input order), salt, metadata.
    Rows inside each split are shuffled with default_rng(seed + 17171) when order is requested."""
    unique_hashes, inverse, counts = np.unique(np.asarray(keys), return_inverse=True, return_counts=True)
    n_families = int(np.asarray(family_codes).max()) + 1
    salts = (np.random.default_rng(seed).integers(1, np.iinfo(np.uint64).max, size=n_candidates, dtype=np.uint64)
             if salt is None else np.asarray([salt], dtype=np.uint64))
    best = best_assignment = best_salt = None
    candidates = []
    for i, cand in enumerate(salts):
        assignment = assign_groups(unique_hashes, cand)
        m = assignment_objective(assignment, inverse, counts, family_codes, n_families)
        candidates.append(dict(candidate=i, salt=str(int(cand)), **m))
        if best is None or m['objective'] < best['objective']:   # first minimum wins
            best, best_assignment, best_salt = m, assignment, int(cand)
    part = best_assignment[inverse]
    meta = dict(seed=int(seed), n_groups=int(len(unique_hashes)), candidate_salts_tested=int(len(salts)),
                selected_salt=str(best_salt), **{f'selected_{k}': v for k, v in best.items()})
    return part, best_salt, meta, candidates


def split_order(part, seed):
    """Row indices of train/val/test, shuffled inside each split as in notebook 01."""
    rng = np.random.default_rng(seed + SHUFFLE_SEED_OFFSET)
    return tuple(rng.permutation(np.flatnonzero(part == i)) for i in range(3))


def random_row_split(family_codes, seed):
    """Random-row baseline of notebook 01: stratified by family, 70/10/20, one generator per seed."""
    codes = np.asarray(family_codes)
    order = np.argsort(codes, kind='stable')
    counts = np.bincount(codes.astype(np.int64))
    offsets = np.r_[0, np.cumsum(counts)]
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for i in range(len(counts)):
        perm = rng.permutation(order[offsets[i]:offsets[i + 1]].astype(np.int64))
        c1, c2 = int(len(perm) * TRAIN_FRAC), int(len(perm) * (TRAIN_FRAC + VAL_FRAC))
        tr.append(perm[:c1]); va.append(perm[c1:c2]); te.append(perm[c2:])
    part = np.empty(len(codes), dtype=np.int8)
    part[np.concatenate(tr)] = 0; part[np.concatenate(va)] = 1; part[np.concatenate(te)] = 2
    return part


def split_audit(keys, y, fam, names, part):
    """Rows, groups, attack rate, overlap in both directions, absent families, largest test group."""
    out = {}
    uniq = [np.unique(keys[part == i]) for i in range(3)]
    for i, s in enumerate(SPLIT_NAMES):
        m = part == i
        out[f'{s}_rows'] = int(m.sum()); out[f'{s}_groups'] = int(len(uniq[i]))
        out[f'{s}_attack_pct'] = float(100 * y[m].mean()) if m.any() else float('nan')
        out[f'{s}_families_absent'] = ';'.join(names[f] for f in range(len(names)) if not (fam[m] == f).any())
    for (a, b) in [(0, 2), (0, 1), (1, 2)]:
        src, dst = SPLIT_NAMES[a], SPLIT_NAMES[b]
        m = part == b
        seen = np.isin(keys[m], uniq[a])
        out[f'{src}_to_{dst}_row_overlap_pct'] = float(100 * seen.mean()) if m.any() else float('nan')
        out[f'{src}_to_{dst}_group_overlap_pct'] = float(100 * np.isin(uniq[b], uniq[a]).mean()) if len(uniq[b]) else float('nan')
    te = part == 2
    if te.any():
        _, c = np.unique(keys[te], return_counts=True)
        out['test_largest_group_rows'] = int(c.max())
    return out


def split_fingerprint(part, row_index=None):
    h = hashlib.sha256(part.astype(np.int8).tobytes())
    if row_index is not None:
        h.update(np.asarray(row_index, dtype=np.int64).tobytes())
    return h.hexdigest()[:20]


# ----------------------------------------------------------- thresholds ----
def _curve(y, s):
    order = np.argsort(-s, kind='stable')
    ys, ss = y[order].astype(np.int64), s[order]
    tp, fp = np.cumsum(ys), np.cumsum(1 - ys)
    last = np.r_[ss[1:] != ss[:-1], True]       # last row of each distinct score block
    tp, fp, thr = tp[last], fp[last], ss[last]
    P = int(ys.sum()); N = len(ys) - P
    return tp, fp, thr, P, N


def best_f1_macro_threshold(y, s):
    """Prediction is score >= threshold. Exact search over distinct validation scores plus the
    all-benign rule (threshold = +inf). Ties are broken toward the largest threshold."""
    tp, fp, thr, P, N = _curve(y, s)
    fn, tn = P - tp, N - fp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = 0.5 * (np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
                    + np.where(2 * tn + fn + fp > 0, 2 * tn / np.maximum(2 * tn + fn + fp, 1), 0.0))
    f1_none = 0.5 * (0.0 + (2 * N / (2 * N + P) if N + P else 0.0))     # predict all benign
    best = int(np.argmax(f1))                                          # first max = largest threshold
    if f1_none > f1[best]:
        return float('inf'), float(f1_none)
    return float(thr[best]), float(f1[best])


def threshold_at_fpr(y, s, target_fpr):
    """Largest-TPR threshold among those with validation FPR <= target (fraction); ties -> largest
    threshold; +inf if no threshold qualifies. Returns threshold and achieved validation FPR/TPR."""
    tp, fp, thr, P, N = _curve(y, s)
    fpr = fp / max(N, 1); tpr = tp / max(P, 1)
    ok = fpr <= target_fpr
    if not ok.any():
        return float('inf'), 0.0, 0.0
    best = int(np.flatnonzero(ok)[np.argmax(tpr[ok])])                 # first max = largest thr
    return float(thr[best]), float(fpr[best]), float(tpr[best])


# -------------------------------------------------------------- metrics ----
def metrics(y, s, pred, w=None):
    """Counts and rates; w = per-row weights (None = each row counts 1). Undefined ratios are NaN."""
    y = y.astype(np.int64); pred = pred.astype(np.int64)
    w = np.ones(len(y)) if w is None else np.asarray(w, dtype=np.float64)
    tp = float(w[(pred == 1) & (y == 1)].sum()); fp = float(w[(pred == 1) & (y == 0)].sum())
    fn = float(w[(pred == 0) & (y == 1)].sum()); tn = float(w[(pred == 0) & (y == 0)].sum())
    div = lambda a, b: a / b if b > 0 else float('nan')
    f1_att = div(2 * tp, 2 * tp + fp + fn); f1_ben = div(2 * tn, 2 * tn + fn + fp)
    out = dict(n=float(w.sum()), n_attack=tp + fn, n_benign=fp + tn, attack_pct=100 * div(tp + fn, w.sum()),
               f1_macro=0.5 * (f1_att + f1_ben), f1_attack=f1_att, fpr_pct=100 * div(fp, fp + tn),
               tpr=div(tp, tp + fn), precision=div(tp, tp + fp), tp=tp, fp=fp, fn=fn, tn=tn)
    out['pr_auc'] = (float(average_precision_score(y, s, sample_weight=w)) if s is not None and 0 < tp + fn < w.sum()
                     else float('nan'))
    return out


def group_balanced_weights(keys_eval):
    """Weight 1 / (group size inside the evaluation set): every group sums to 1."""
    _, inv, cnt = np.unique(keys_eval, return_inverse=True, return_counts=True)
    return 1.0 / cnt[inv]


def family_recall(y, pred, fam, names, min_rows=1):
    rows = []
    for f in range(len(names)):
        m = (fam == f) & (y == 1)
        n = int(m.sum())
        rows.append(dict(family=str(names[f]), count=n, recall=(float(pred[m].mean()) if n >= min_rows else float('nan'))))
    return rows


def evaluate_run(y_te, s_te, keys_te, fam_te, names, y_va, s_va, fpr_targets=(0.001, 0.0001)):
    """Everything derived from saved scores: F1-macro threshold, FPR-target thresholds,
    row-weighted and group-balanced metrics, per-family recall."""
    thr, f1_val = best_f1_macro_threshold(y_va, s_va)
    pred = s_te >= thr
    out = dict(threshold=thr, val_f1_macro_at_threshold=f1_val)
    out.update(metrics(y_te, s_te, pred))
    gw = group_balanced_weights(keys_te)
    out.update({f'gb_{k}': v for k, v in metrics(y_te, s_te, pred, gw).items()
                if k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr', 'precision')})
    for t in fpr_targets:
        tag = f'fpr{t * 100:g}'
        n_ben_val = int((y_va == 0).sum())
        th, vf, vt = threshold_at_fpr(y_va, s_va, t)
        p = s_te >= th
        m = metrics(y_te, None, p)
        out.update({f'{tag}_threshold': th, f'{tag}_val_fpr_pct': 100 * vf, f'{tag}_val_tpr': vt,
                    f'{tag}_val_benign_rows': n_ben_val, f'{tag}_test_fpr_pct': m['fpr_pct'],
                    f'{tag}_test_fp': m['fp'], f'{tag}_test_tpr': m['tpr'],
                    f'{tag}_interpretable': bool(n_ben_val * t >= 10)})   # at least 10 expected FP on validation
    out['family_recall'] = family_recall(y_te, pred, fam_te, names)
    return out, pred


def minmax_fit(Xtr):
    mn, mx = Xtr.min(0), Xtr.max(0)
    rng_ = np.where(mx > mn, mx - mn, 1.0).astype(np.float32)
    return mn, rng_


def minmax_apply(X, mn, rng_, clip=None):
    Z = (X - mn) / rng_
    if clip is not None:
        np.clip(Z, clip[0], clip[1], out=Z)
    return Z


# ------------------------------------------------------------ bootstrap ----
def paired_group_bootstrap(keys, y, scores, n_boot=2000, seed=20261111, group_balanced=False):
    """Paired bootstrap over groups of one fixed evaluation set.
    scores: dict condition -> (score array, threshold). The same group multiplicities are
    applied to every condition in each draw. Returns per-draw metrics (dict of arrays) and the number
    of draws in which a class was missing (those draws are left as NaN)."""
    uniq, inv, cnt = np.unique(keys, return_inverse=True, return_counts=True)
    G = len(uniq)
    rng = np.random.default_rng(seed)
    base_w = (1.0 / cnt[inv]) if group_balanced else np.ones(len(keys))
    conds = list(scores)
    preds = {c: (scores[c][0] >= scores[c][1]) for c in conds}
    y64 = y.astype(np.int64)
    store = {c: {k: np.full(n_boot, np.nan) for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')} for c in conds}
    invalid = 0
    for b in range(n_boot):
        mult = np.bincount(rng.integers(0, G, G), minlength=G)
        w = base_w * mult[inv]
        if w[y64 == 1].sum() == 0 or w[y64 == 0].sum() == 0:
            invalid += 1
            continue
        for c in conds:
            m = metrics(y64, scores[c][0], preds[c], w)
            for k in store[c]:
                store[c][k][b] = m[k]
    return store, invalid


def bootstrap_delta_table(store, pairs, invalid, n_boot):
    rows = []
    for a, b in pairs:
        for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr'):
            d = store[a][k] - store[b][k]
            d = d[~np.isnan(d)]
            rows.append(dict(comparison=f'{a} - {b}', metric=k, n_valid=int(len(d)), n_invalid=int(invalid),
                             delta_boot_mean=float(d.mean()) if len(d) else float('nan'),
                             ci95_low=float(np.percentile(d, 2.5)) if len(d) else float('nan'),
                             ci95_high=float(np.percentile(d, 97.5)) if len(d) else float('nan')))
    return pd.DataFrame(rows)


In [4]:
# MLP variant of the paper (graph encoder and gate removed, no decoder): encoder 39 -> 192 -> 96,
# classifier on [x || z] (135) -> 192 -> 96 -> 1, focal BCE, AdamW, early stopping on validation F1-macro.
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_OK = True
except ImportError:
    TORCH_OK = False

if TORCH_OK:
    class MLPDetector(nn.Module):
        def __init__(self, d_in=N_FEATURES, cfg=MLP_CFG):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d_in, cfg['enc_hidden']), nn.ReLU(), nn.Dropout(cfg['enc_dropout']),
                                     nn.Linear(cfg['enc_hidden'], cfg['latent']), nn.ReLU())
            h1, h2 = cfg['cls_hidden']
            self.cls = nn.Sequential(nn.Linear(d_in + cfg['latent'], h1), nn.ReLU(), nn.Dropout(cfg['cls_dropout']),
                                     nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, 1))
        def forward(self, x):
            z = self.enc(x)
            return self.cls(torch.cat([x, z], dim=1)).squeeze(1)

    def focal_bce(logits, y, alpha, gamma, pos_weight):
        bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * y + (1 - p) * (1 - y)
        a_t = alpha * y + (1 - alpha) * (1 - y)
        return (a_t * (1 - p_t) ** gamma * bce).mean()

    @torch.no_grad()
    def mlp_scores(model, X, batch=20000):
        model.eval(); out = []
        for i in range(0, len(X), batch):
            out.append(torch.sigmoid(model(torch.as_tensor(X[i:i + batch], device=DEVICE))).cpu().numpy())
        return np.concatenate(out).astype(np.float32)

    def train_mlp(Xtr, ytr, Xva, yva, seed, cfg=MLP_CFG):
        torch.manual_seed(seed); np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        model = MLPDetector(cfg=cfg).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        pw = torch.tensor([min(float((ytr == 0).sum()) / max(int((ytr == 1).sum()), 1), cfg['pos_weight_cap'])],
                          device=DEVICE)
        rng = np.random.default_rng(seed)
        best_f1, best_state, best_epoch, bad, history = -1.0, None, 0, 0, []
        Xtr_t = torch.as_tensor(Xtr); ytr_t = torch.as_tensor(ytr.astype(np.float32))
        for epoch in range(1, cfg['max_epochs'] + 1):
            model.train(); perm = rng.permutation(len(Xtr)); tot = 0.0; nb = 0
            for i in range(0, len(perm), cfg['batch']):
                b = perm[i:i + cfg['batch']]
                xb = Xtr_t[b].to(DEVICE, non_blocking=True); yb = ytr_t[b].to(DEVICE, non_blocking=True)
                loss = focal_bce(model(xb), yb, cfg['focal_alpha'], cfg['focal_gamma'], pw)
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip']); opt.step()
                tot += float(loss); nb += 1
            s_va = mlp_scores(model, Xva)
            thr, f1 = best_f1_macro_threshold(yva, s_va)
            history.append(dict(epoch=epoch, train_loss=tot / max(nb, 1), val_f1_macro=f1, val_threshold=thr))
            if f1 > best_f1:
                best_f1, best_epoch, bad = f1, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg['patience']:
                    break
        model.load_state_dict(best_state)
        return model, dict(best_epoch=best_epoch, epochs_run=epoch, best_val_f1_macro=best_f1,
                           pos_weight=float(pw.item()), n_params=sum(p.numel() for p in model.parameters())), history

In [5]:
def cohort_rows(data, cohort):
    if cohort == 'full':
        return np.arange(len(data['y']), dtype=np.int64)
    return load_frozen_cohort(data['name'], data, SUBSET_ROOT)


def build_split(data, cohort, rows, protocol, split_seed):
    """Returns part (0/1/2 per cohort row, in cohort order) and metadata; saves both."""
    tag = f'{data["name"]}__{cohort}__{protocol}__split{split_seed}'
    path = os.path.join(OUT_DIR, 'splits', tag + '.npz')
    meta_path = os.path.join(OUT_DIR, 'splits', tag + '.json')
    keys, y, fam = data['keys'][rows], data['y'][rows], data['fam'][rows]
    if os.path.exists(path):
        z = np.load(path); part = z['part']
        assert len(part) == len(rows) and np.array_equal(z['row_index'], rows), 'stored split does not match the cohort'
        return part, json.load(open(meta_path))
    meta = dict(dataset=data['name'], cohort=cohort, protocol=protocol, split_seed=int(split_seed),
                cohort_rows=int(len(rows)), cohort_groups=int(len(np.unique(keys))))
    if protocol == 'random_row':
        part = random_row_split(fam, split_seed)
        meta['method'] = 'random_row_split of notebook 01 (one generator per seed, stratified by family)'
    else:
        ref = PAPER_FULL_SPLITS.get((data['name'], split_seed))
        if cohort == 'full' and FULL_GD_FROM_PAPER_SALTS and ref is not None:
            part, salt, m, cands = hash_group_split(keys, fam, split_seed, salt=ref['salt'])
            for i, s in enumerate(SPLIT_NAMES):
                got = (int((part == i).sum()), int(((part == i) & (y == 0)).sum()), int(((part == i) & (y == 1)).sum()))
                assert got == tuple(ref[s]), f'{tag}: {s} counts {got} differ from the stored split {tuple(ref[s])}'
            meta['method'] = 'Algorithm 1 with the recorded salt of the paper (no candidate search)'
        else:
            part, salt, m, cands = hash_group_split(keys, fam, split_seed, n_candidates=N_CANDIDATES_NEW_SPLITS)
            meta['method'] = f'Algorithm 1, candidate search over {N_CANDIDATES_NEW_SPLITS} salts'
            pd.DataFrame(cands).to_csv(os.path.join(OUT_DIR, 'splits', tag + '_candidates.csv'), index=False)
        meta.update(m)
        for a, b in [(0, 1), (0, 2), (1, 2)]:
            assert not np.intersect1d(np.unique(keys[part == a]), np.unique(keys[part == b])).size, 'GD split shares a group'
    for i, s in enumerate(SPLIT_NAMES):
        if not ((part == i) & (y == 1)).any() or not ((part == i) & (y == 0)).any():
            raise RuntimeError(f'{tag}: split {s} lacks a binary class; investigate, do not try other seeds silently')
    meta.update(split_audit(keys, y, fam, data['names'], part))
    meta['fingerprint'] = split_fingerprint(part, rows)
    np.savez_compressed(path, part=part, row_index=rows, key=keys)
    atomic_json(meta, meta_path)
    return part, meta

In [6]:
def run_id(ds, cohort, protocol, split_seed, model, model_seed):
    return f'{ds}__{cohort}__{protocol}__split{split_seed}__{model}__model{model_seed}'


def fit_and_score(model, Xtr, ytr, Xva, yva, Xte, model_seed):
    t0 = time.time()
    if model == 'lightgbm':
        import lightgbm as lgb
        spw = min(float((ytr == 0).sum()) / max(int((ytr == 1).sum()), 1), 12.0)
        clf = lgb.LGBMClassifier(**LGBM_PARAMS, scale_pos_weight=spw, random_state=model_seed,
                                 bagging_seed=model_seed, feature_fraction_seed=model_seed, data_random_seed=model_seed)
        clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='binary_logloss',
                callbacks=[lgb.early_stopping(LGBM_EARLY_STOP, verbose=False)])
        fit_s = time.time() - t0; t1 = time.time()
        s_va = clf.predict_proba(Xva)[:, 1].astype(np.float32); s_te = clf.predict_proba(Xte)[:, 1].astype(np.float32)
        info = dict(best_iteration=int(clf.best_iteration_ or LGBM_PARAMS['n_estimators']),
                    reached_ceiling=bool((clf.best_iteration_ or 0) >= LGBM_PARAMS['n_estimators']),
                    scale_pos_weight=spw, config=LGBM_PARAMS)
    elif model == 'mlp':
        assert TORCH_OK, 'PyTorch is not available'
        net, info, history = train_mlp(Xtr, ytr, Xva, yva, model_seed)
        fit_s = time.time() - t0; t1 = time.time()
        s_va = mlp_scores(net, Xva); s_te = mlp_scores(net, Xte)
        info['config'] = MLP_CFG; info['history'] = history
    else:
        raise ValueError(model)
    return s_va, s_te, info, fit_s, time.time() - t1


def run_one(data, cohort, rows, protocol, split_seed, part, split_meta, model, model_seed):
    rid = run_id(data['name'], cohort, protocol, split_seed, model, model_seed)
    res_path = os.path.join(OUT_DIR, 'runs', rid + '.json')
    if os.path.exists(res_path):
        return json.load(open(res_path))
    tr, va, te = (rows[part == i] for i in range(3))
    tr_order = np.random.default_rng(split_seed + SHUFFLE_SEED_OFFSET).permutation(len(tr)); tr = tr[tr_order]
    X, y = data['X'], data['y']
    mn, rng_ = minmax_fit(X[tr])
    Xtr = minmax_apply(X[tr], mn, rng_, CLIP); Xva = minmax_apply(X[va], mn, rng_, CLIP); Xte = minmax_apply(X[te], mn, rng_, CLIP)
    out_of_range = float(((X[te] < mn) | (X[te] > mn + rng_)).mean())
    s_va, s_te, info, fit_s, pred_s = fit_and_score(model, Xtr, y[tr], Xva, y[va], Xte, model_seed)
    ev, pred = evaluate_run(y[te], s_te, data['keys'][te], data['fam'][te], data['names'], y[va], s_va, FPR_TARGETS)
    np.savez_compressed(os.path.join(OUT_DIR, 'predictions', rid + '.npz'),
                        row_index=te, key=data['keys'][te], y_true=y[te], family=data['fam'][te],
                        score=s_te, y_pred=pred.astype(np.int8), threshold=np.float64(ev['threshold']),
                        val_row_index=va, val_score=s_va, val_y_true=y[va], family_names=data['names'])
    result = dict(run_id=rid, dataset=data['name'], cohort=cohort, protocol=protocol, split_seed=split_seed,
                  model=model, model_seed=model_seed, split_fingerprint=split_meta['fingerprint'],
                  split_salt=split_meta.get('selected_salt'), clip=CLIP, test_out_of_train_range_frac=out_of_range,
                  fit_seconds=round(fit_s, 1), predict_seconds=round(pred_s, 1), smoke=SMOKE,
                  **{k: v for k, v in info.items() if k not in ('history', 'config')}, **ev)
    result['model_config'] = info.get('config'); result['history'] = info.get('history')
    atomic_json(result, res_path)
    log(f'{rid}: F1 {ev["f1_macro"]:.4f} | AP {ev["pr_auc"]:.4f} | FPR {ev["fpr_pct"]:.4f}% | TPR {ev["tpr"]:.4f} '
        f'| fit {fit_s:.0f}s')
    return result


for ds in DATASETS_TO_RUN:
    data = None
    for cohort, models in PLAN.items():
        todo = [(p, ss, m, ms) for p in PROTOCOLS for ss in SPLIT_SEEDS for m in models for ms in MODEL_SEEDS
                if not os.path.exists(os.path.join(OUT_DIR, 'runs', run_id(ds, cohort, p, ss, m, ms) + '.json'))]
        if not todo:
            log(f'{ds}/{cohort}: all runs done'); continue
        if data is None:
            data = load_dataset(ds, DATA_DIR)
            atomic_json(dict(dataset=ds, feature_order=data['feat_cols'], replaced_values_per_column=data['replaced_per_column'],
                             rows=int(len(data['y'])), groups=data['n_groups']), os.path.join(OUT_DIR, 'reports', f'{ds}_manifest.json'))
        rows = cohort_rows(data, cohort)
        for p in PROTOCOLS:
            for ss in SPLIT_SEEDS:
                part, meta = build_split(data, cohort, rows, p, ss)
                for m in models:
                    for ms in MODEL_SEEDS:
                        if (p, ss, m, ms) in todo:
                            run_one(data, cohort, rows, p, ss, part, meta, m, ms); gc.collect()
    del data; gc.collect()

14:05:30 UNSW_V2/frozen: all runs done
14:05:30 UNSW_V2/full: all runs done
14:05:30 CIC_V2/frozen: all runs done


/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 1,000,000 rows read (6s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 2,000,000 rows read (11s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


... (29 outputs left out)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 18,000,000 rows read (83s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 18,893,708 rows read (87s)
14:07:08 CIC_V2: groups 8,981,062 (audit 8,981,062) | repeated-row rate 52.47% | attack 11.95% | replaced values 255
14:15:50 CIC_V2__full__hash_group_disjoint__split42__lightgbm__model456: F1 0.9891 | AP 0.9806 | FPR 0.0194% | TPR 0.9638 | fit 440s
14:23:17 CIC_V2__full__hash_group_disjoint__split123__lightgbm__model42: F1 0.9766 | AP 0.9826 | FPR 0.0100% | TPR 0.9216 | fit 296s
14:28:06 CIC_V2__full__hash_group_disjoint__split123__lightgbm__model123: F1 0.9763 | AP 0.9824 | FPR 0.0278% | TPR 0.9219 | fit 252s
14:34:20 CIC_V2__full__hash_group_disjoint__split123__lightgbm__model456: F1 0.9764 | AP 0.9826 | FPR 0.0303% | TPR 0.9225 | fit 327s
14:45:06 CIC_V2__full__hash_group_disjoint__split456__lightgbm__model42: F1 0.9877 | AP 0.9786 | FPR 0.0181% | TPR 0.9584 | fit 466s
14:54:19 CIC_V2__full__hash_group_disjoint__split456__lightgbm__model123: F1 0.9877 | AP 0.9786 | FPR 0.0194% | TPR 0.9586 | fit 481s
15:04:10 CIC_V2__full__hash_group_disjoint__spl

In [7]:
runs = [json.load(open(os.path.join(OUT_DIR, 'runs', f))) for f in sorted(os.listdir(os.path.join(OUT_DIR, 'runs')))]
fam_rows = [dict(run_id=r['run_id'], dataset=r['dataset'], cohort=r['cohort'], protocol=r['protocol'],
                 split_seed=r['split_seed'], model=r['model'], model_seed=r['model_seed'], **f)
            for r in runs for f in r['family_recall']]
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('family_recall', 'history', 'model_config')} for r in runs])
df.to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_all_runs.csv'), index=False)
pd.DataFrame(fam_rows).to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_family_recall_runs.csv'), index=False)

split_meta = [json.load(open(os.path.join(OUT_DIR, 'splits', f))) for f in sorted(os.listdir(os.path.join(OUT_DIR, 'splits')))
              if f.endswith('.json')]
pd.DataFrame(split_meta).to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_split_audit.csv'), index=False)

METRICS = ['f1_macro', 'pr_auc', 'fpr_pct', 'tpr', 'precision', 'gb_f1_macro', 'gb_pr_auc', 'gb_fpr_pct', 'gb_tpr',
           'fpr0.1_test_tpr', 'fpr0.1_test_fpr_pct', 'fpr0.1_test_fp']
keyc = ['dataset', 'cohort', 'model', 'protocol']
# level 1: mean/std over model seeds inside each split seed; level 2: mean of split means, std across split seeds
lvl1 = df.groupby(keyc + ['split_seed'])[METRICS].agg(['mean', 'std'])
lvl1.columns = [f'{a}_{b}' for a, b in lvl1.columns]
lvl1 = lvl1.reset_index()
lvl2 = lvl1.groupby(keyc)[[f'{m}_mean' for m in METRICS]].agg(['mean', 'std'])
lvl2.columns = [f'{a.replace("_mean", "")}_{"mean_of_split_means" if b == "mean" else "std_across_splits"}' for a, b in lvl2.columns]
lvl2 = lvl2.reset_index()
lvl1.to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_level1_by_split_seed.csv'), index=False)
lvl2.to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_level2_summary.csv'), index=False)

# delta_A = RR - GD for each (split_seed, model_seed) pair of the same dataset/cohort/model
piv = df.pivot_table(index=['dataset', 'cohort', 'model', 'split_seed', 'model_seed'], columns='protocol', values=METRICS)
delta = pd.DataFrame({m: piv[(m, 'random_row')] - piv[(m, 'hash_group_disjoint')] for m in METRICS
                      if (m, 'random_row') in piv and (m, 'hash_group_disjoint') in piv}).reset_index()
delta.to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_delta_rr_minus_gd_per_pair.csv'), index=False)
dsum = delta.groupby(['dataset', 'cohort', 'model']).agg({m: ['mean', 'std', 'min', 'max', lambda s: float((s > 0).mean())]
                                                           for m in METRICS if m in delta})
dsum.columns = [f'{a}_{b if not b.startswith("<") else "frac_positive"}' for a, b in dsum.columns]
dsum.to_csv(os.path.join(OUT_DIR, 'reports', 'stepA_delta_summary.csv'))

pd.set_option('display.width', 220)
print('Split audit (overlap train->test, rows / groups, %):')
print(pd.DataFrame(split_meta)[['dataset', 'cohort', 'protocol', 'split_seed', 'train_rows', 'validation_rows', 'test_rows',
                                'test_attack_pct', 'train_to_test_row_overlap_pct', 'train_to_test_group_overlap_pct',
                                'train_to_validation_row_overlap_pct', 'test_families_absent']].round(2).to_string(index=False))
print('\nLevel-2 summary (mean of split means; std across split seeds):')
print(lvl2[keyc + [f'{m}_mean_of_split_means' for m in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')]
           + [f'{m}_std_across_splits' for m in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')]].round(4).to_string(index=False))
print('\nRR - GD per pair (mean, std, min, max, fraction positive):')
print(dsum[[c for c in dsum.columns if c.split('_')[0] in ('f1', 'pr', 'fpr', 'tpr') and 'gb' not in c]].round(4).to_string())
print('\nOperating point at validation-selected FPR <= 0.1 %: test TPR, achieved test FPR, FP count')
print(lvl2[keyc + ['fpr0.1_test_tpr_mean_of_split_means', 'fpr0.1_test_fpr_pct_mean_of_split_means', 'fpr0.1_test_fp_mean_of_split_means']].round(4).to_string(index=False))

Split audit (overlap train->test, rows / groups, %):
dataset cohort            protocol  split_seed  train_rows  validation_rows  test_rows  test_attack_pct  train_to_test_row_overlap_pct  train_to_test_group_overlap_pct  train_to_validation_row_overlap_pct     test_families_absent
 CIC_V2 frozen hash_group_disjoint         123      352452            40241     103691            28.93                           0.00                             0.00                                 0.00    DoS attacks-Slowloris
 CIC_V2 frozen hash_group_disjoint          42      347706            44731     103947            27.25                           0.00                             0.00                                 0.00                         
 CIC_V2 frozen hash_group_disjoint         456      353441            51141      91802            28.40                           0.00                             0.00                                 0.00 DoS attacks-SlowHTTPTest
 CIC_V2 frozen          ran